In [ ]:
pip install pandas openpyxl

In [1]:
import os
import re
import pandas as pd
from pathlib import Path

def replace_umlauts(text):
    """Replace German umlauts with their English equivalents."""
    replacements = {
        'ä': 'ae', 'Ä': 'Ae',
        'ö': 'oe', 'Ö': 'Oe',
        'ü': 'ue', 'Ü': 'Ue',
        'ß': 'ss'
    }
    for german, english in replacements.items():
        text = text.replace(german, english)
    return text

def clean_filename(text):
    """Clean filename by replacing umlauts, spaces, and removing ALL symbols."""
    # Replace umlauts first
    text = replace_umlauts(text)
    
    # Replace common separators with hyphens before removing them entirely
    # This helps maintain word separation
    text = text.replace(' ', '-')
    text = text.replace('_', '-')
    text = text.replace('/', '-')
    text = text.replace('\\', '-')
    text = text.replace(':', '-')
    text = text.replace(',', '-')
    text = text.replace(';', '-')
    text = text.replace('.', '-')  # Replace dots too (except for file extension)
    text = text.replace('(', '-')
    text = text.replace(')', '-')
    text = text.replace('[', '-')
    text = text.replace(']', '-')
    text = text.replace('{', '-')
    text = text.replace('}', '-')
    
    # Remove ALL remaining symbols - keep only letters, numbers, and hyphens
    # This catches any symbol we might have missed above
    text = re.sub(r'[^a-zA-Z0-9\-]', '', text)
    
    # Clean up multiple consecutive hyphens
    text = re.sub(r'-+', '-', text)
    
    # Remove leading/trailing hyphens
    text = text.strip('-')
    
    return text

def rename_images():
    """Main function to rename images based on Excel data."""
    
    # File paths
    excel_file = 'lanordica-2035.xlsx'
    
    # Check if Excel file exists
    if not os.path.exists(excel_file):
        print(f"Error: Excel file '{excel_file}' not found!")
        return
    
    try:
        # Read Excel file
        print(f"Reading Excel file: {excel_file}")
        df = pd.read_excel(excel_file)
        
        # Check if required columns exist
        if 'Artikelnummer' not in df.columns or 'Produktname' not in df.columns:
            print("Error: Excel file must contain 'Artikelnummer' and 'Produktname' columns!")
            return
        
        # Remove rows with NaN values in required columns
        df = df.dropna(subset=['Artikelnummer', 'Produktname'])
        
        # Convert Artikelnummer to string and strip whitespace
        df['Artikelnummer'] = df['Artikelnummer'].astype(str).str.strip()
        df['Produktname'] = df['Produktname'].astype(str).str.strip()
        
        print(f"Found {len(df)} products in Excel file")
        
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return
    
    # Ask for folder path
    folder_path = input("Enter the folder path containing images: ").strip()
    
    # Remove quotes if present
    folder_path = folder_path.strip('"').strip("'")
    
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found!")
        return
    
    # Get all files in the folder
    files = os.listdir(folder_path)
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}
    
    # Filter only image files
    image_files = [f for f in files if Path(f).suffix.lower() in image_extensions]
    
    if not image_files:
        print("No image files found in the specified folder!")
        return
    
    print(f"Found {len(image_files)} image files")
    
    # Create a dictionary for quick lookup
    artikel_dict = {}
    for _, row in df.iterrows():
        artikel_dict[str(row['Artikelnummer'])] = row['Produktname']
    
    # Counter for renamed files
    renamed_count = 0
    skipped_count = 0
    
    # Process each image file
    for image_file in image_files:
        file_renamed = False
        
        # Check each Artikelnummer to see if the image starts with it
        for artikelnummer, produktname in artikel_dict.items():
            if image_file.startswith(artikelnummer):
                # Get file extension
                file_ext = Path(image_file).suffix
                
                # Get filename without extension
                filename_no_ext = os.path.splitext(image_file)[0]
                
                # Check if there's already a number suffix at the end (e.g., -1, -2, -3)
                # This regex looks for a hyphen followed by one or more digits at the end
                number_suffix_match = re.search(r'-(\d+)$', filename_no_ext)
                
                # Clean the product name and artikelnummer
                clean_produktname = clean_filename(produktname)
                clean_artikelnummer = clean_filename(artikelnummer)
                
                if number_suffix_match:
                    # Keep the existing number suffix
                    number_suffix = number_suffix_match.group(1)
                    new_name = f"{clean_artikelnummer}-{clean_produktname}-{number_suffix}{file_ext}"
                else:
                    # No number suffix, just use Artikelnummer-Produktname
                    new_name = f"{clean_artikelnummer}-{clean_produktname}{file_ext}"
                
                # Full paths
                old_path = os.path.join(folder_path, image_file)
                new_path = os.path.join(folder_path, new_name)
                
                # Check if new filename already exists
                if os.path.exists(new_path) and old_path != new_path:
                    print(f"Warning: '{new_name}' already exists. Skipping '{image_file}'")
                    skipped_count += 1
                else:
                    try:
                        os.rename(old_path, new_path)
                        print(f"Renamed: {image_file} -> {new_name}")
                        renamed_count += 1
                        file_renamed = True
                    except Exception as e:
                        print(f"Error renaming '{image_file}': {e}")
                        skipped_count += 1
                
                break
        
        if not file_renamed:
            print(f"No matching Artikelnummer found for: {image_file}")
            skipped_count += 1
    
    # Summary
    print("\n" + "="*50)
    print(f"Renaming complete!")
    print(f"Files renamed: {renamed_count}")
    print(f"Files skipped: {skipped_count}")
    print(f"Total processed: {len(image_files)}")

if __name__ == "__main__":
    try:
        rename_images()
    except KeyboardInterrupt:
        print("\n\nOperation cancelled by user.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
    
    input("\nPress Enter to exit...")

Reading Excel file: lanordica-2035.xlsx
Found 12 products in Excel file
Found 102 image files
Renamed: 1291403-1.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-1.jpg
Renamed: 1291403-2.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-2.jpg
Renamed: 1291403-3.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-3.jpg
Renamed: 1291403-4.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-4.jpg
Renamed: 1291403-5.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-5.jpg
Renamed: 1291403-6.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen-6.jpg
Renamed: 1291403-EEL.jpg -> 1291403-Extraflame-Prestige-Line-DEBBY-16-EVO-NERO-Pelletofen.jpg
Renamed: 1295251-1.jpg -> 1295251-Extraflame-TEOREMA-PLUS-5-0-Evo-AVOR-Pelletofen-1.jpg
Renamed: 1295251-2.jpg -> 1295251-Extraflame-TEOREMA-PLUS-5-0-Evo-AVOR-Pelletofen-2.jpg
Renamed: 1295251-3.jpg -> 1295251-Extraflame-TEOREMA-PLUS-5-0-Evo-AVOR-P